# ImageNet Training Notebook (Kaggle)

Train FracBNN / Ada-FracBNN on an ImageNet-style Kaggle subset with three model variants:

| model\_id | Architecture | Description |
|-----------|-------------|-------------|
| 0 | `binput-pg-quant-shortcut` | Baseline PG |
| 1 | `adaptive-pg` | Adaptive PG (Ada-FracBNN) |
| 2 | `adaptive-pg-kd` | Adaptive PG + Knowledge Distillation |

## Before you start
1. **Add the dataset**: Click **Add data** and attach `amitpant7/imagenet1k-subset-100k-train-and-10k-val`.
2. **Enable GPU**: Settings > Accelerator > **GPU T4 x2** (or P100).
3. **Enable Internet**: Settings > Internet > **On** (needed for git clone and pip install).
4. **Persistence**: Set Settings > Persistence > **Files only** so `/kaggle/working/` survives restarts.

### Important note
This dataset is a **subset**, not full ImageNet-1K. It is good for validating the training pipeline and comparing the new changes, but the resulting top-1 accuracy will **not** be comparable to full ImageNet numbers from the paper.

### Kaggle paths
- `/kaggle/input/` -- read-only attached datasets
- `/kaggle/working/` -- writable output directory (~70 GB); checkpoints go here
- Session limit: 12 h with GPU

## Cell 1 -- GPU and Disk Check

In [ ]:
!nvidia-smi
import torch, os
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"VRAM: {mem_gb:.1f} GB")
    print(f"GPU count: {torch.cuda.device_count()}")
!df -h /kaggle/working

## Cell 2 -- Clone Repo

In [ ]:
import os

REPO_DIR = "/kaggle/working/endingengineering"
REPO_URL = "https://github.com/YOUR_USERNAME/endingengineering.git"  # <-- replace
BRANCH   = "feat/imagenet-colab-bootstrap"

if not os.path.exists(REPO_DIR):
    !git clone --branch $BRANCH $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git fetch origin $BRANCH && git checkout $BRANCH && git pull origin $BRANCH

os.chdir(REPO_DIR)
!git log --oneline -5

## Cell 3 -- Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

## Cell 4 -- Smoke Test (verify ImageNet models load)

In [ ]:
!python test_imagenet_models.py

## Cell 5 -- Prepare the Kaggle ImageNet Subset

This notebook now targets `amitpant7/imagenet1k-subset-100k-train-and-10k-val`.
The prep cell auto-detects a dataset root under `/kaggle/input/` and looks recursively for
`train/` and `val/` directories that already match the `ImageFolder` layout expected by `imagenet.py`.

If the attached dataset has multiple nested folders, the cell will print the candidates it finds and
symlink the detected `train/` and `val/` into `/kaggle/working/imagenet/`.

In [ ]:
import os, pathlib

DATASET_SLUG = "imagenet1k-subset-100k-train-and-10k-val"
KAGGLE_INPUT_ROOT = pathlib.Path("/kaggle/input")
IMAGENET_DIR = pathlib.Path("/kaggle/working/imagenet")


def count_class_dirs(path: pathlib.Path) -> int:
    try:
        return sum(1 for p in path.iterdir() if p.is_dir())
    except (FileNotFoundError, NotADirectoryError, PermissionError):
        return 0


def looks_like_imagefolder_split(path: pathlib.Path) -> bool:
    return path.is_dir() and count_class_dirs(path) >= 10


def candidate_roots(base: pathlib.Path, max_depth: int = 4):
    roots = []
    queue = [(base, 0)]
    seen = set()
    while queue:
        current, depth = queue.pop(0)
        if current in seen or depth > max_depth:
            continue
        seen.add(current)
        roots.append(current)
        if depth == max_depth:
            continue
        try:
            children = [p for p in current.iterdir() if p.is_dir()]
        except (FileNotFoundError, NotADirectoryError, PermissionError):
            continue
        for child in sorted(children):
            queue.append((child, depth + 1))
    return roots


def find_split_dirs_fast(root: pathlib.Path):
    train_candidates = []
    val_candidates = []
    for candidate in candidate_roots(root, max_depth=4):
        for train_name in ("train", "subtrain"):
            train_path = candidate / train_name
            if looks_like_imagefolder_split(train_path):
                train_candidates.append(train_path)
        for val_name in ("val", "valid", "validation", "imagenet_subval"):
            val_path = candidate / val_name
            if looks_like_imagefolder_split(val_path):
                val_candidates.append(val_path)
    return train_candidates, val_candidates


if not KAGGLE_INPUT_ROOT.exists():
    raise FileNotFoundError("/kaggle/input not found. Run this only inside a Kaggle notebook.")

attached_roots = sorted([p for p in KAGGLE_INPUT_ROOT.iterdir() if p.is_dir()])
print("Attached Kaggle datasets:")
for p in attached_roots:
    print(f"  {p.name}")

preferred_roots = [
    KAGGLE_INPUT_ROOT / DATASET_SLUG,
    KAGGLE_INPUT_ROOT / "datasets" / "amitpant7" / DATASET_SLUG,
]
search_root = next((p for p in preferred_roots if p.exists()), None)
if search_root is not None:
    print(f"\nUsing preferred dataset root: {search_root}")
else:
    search_root = KAGGLE_INPUT_ROOT
    print(f"\nPreferred dataset `{DATASET_SLUG}` not found, searching all attached datasets")

train_candidates, val_candidates = find_split_dirs_fast(search_root)

print("\nTrain candidates:")
for p in train_candidates[:10]:
    print(f"  {p} (classes={count_class_dirs(p)})")
print("\nVal candidates:")
for p in val_candidates[:10]:
    print(f"  {p} (classes={count_class_dirs(p)})")

if not train_candidates or not val_candidates:
    raise FileNotFoundError(
        "Could not find ImageFolder-style train/ and val/ directories. "
        "Attach `amitpant7/imagenet1k-subset-100k-train-and-10k-val` and rerun this cell."
    )

train_src = max(train_candidates, key=count_class_dirs)
val_src = max(val_candidates, key=count_class_dirs)

print(f"\nSelected train: {train_src}")
print(f"Selected val:   {val_src}")

IMAGENET_DIR.mkdir(parents=True, exist_ok=True)
for split_name, split_src in [("train", train_src), ("val", val_src)]:
    split_dst = IMAGENET_DIR / split_name
    if split_dst.is_symlink():
        linked_to = pathlib.Path(os.readlink(split_dst))
        if linked_to != split_src:
            split_dst.unlink()
            split_dst.symlink_to(split_src)
            print(f"Updated {split_name} symlink -> {split_src}")
        else:
            print(f"{split_name} symlink already points to {split_src}")
    elif split_dst.exists():
        print(f"{split_name}/ already exists at {split_dst}")
    else:
        split_dst.symlink_to(split_src)
        print(f"Symlinked {split_name} -> {split_src}")

train_classes = count_class_dirs(IMAGENET_DIR / "train")
val_classes = count_class_dirs(IMAGENET_DIR / "val")
print(f"\nTrain classes: {train_classes}, Val classes: {val_classes}")
!du -sh /kaggle/working/imagenet

## Cell 6 -- Set Checkpoint Directory

Everything under `/kaggle/working/` is saved as notebook output and persists across sessions
(when Persistence is set to **Files only**).

In [ ]:
SAVE_DIR = "/kaggle/working/checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Checkpoints will be saved to {SAVE_DIR}")

---
## Cell 7 -- Train Model 0: Baseline PG (`binput-pg-quant-shortcut`)

Full schedule: 120 epochs.  
Kaggle GPU sessions last 12 h -- use Cell 11 to resume across sessions.  
Reduce `-e` for a shorter run.

In [ ]:
os.chdir(REPO_DIR)

!python imagenet.py \
    -id 0 \
    -e  120 \
    -b  128 \
    -lr 5e-4 \
    -wd 1e-5 \
    -d  $IMAGENET_DIR \
    --label_smoothing 0.1 \
    -gpu 0 \
    -s

!cp -a save_ImageNet_model "$SAVE_DIR/save_ImageNet_model_baseline"

## Cell 8 -- Train Model 1: Adaptive PG (`adaptive-pg`)

In [ ]:
!python imagenet.py \
    -id 1 \
    -e  120 \
    -b  128 \
    -lr 5e-4 \
    -wd 1e-5 \
    -d  $IMAGENET_DIR \
    -ts 0.15 \
    -sw 0.01 \
    --label_smoothing 0.1 \
    -gpu 0 \
    -s

!cp -a save_ImageNet_model "$SAVE_DIR/save_ImageNet_model_adaptive"

## Cell 9 -- Train Model 2: Adaptive PG + KD (`adaptive-pg-kd`)

Uses a **pretrained torchvision ResNet-50** as teacher (downloaded automatically).  
Batch size is **64** because both student and teacher must fit in GPU memory.

In [ ]:
!python imagenet.py \
    -id 2 \
    -e  120 \
    -b  64 \
    -lr 5e-4 \
    -wd 1e-5 \
    -d  $IMAGENET_DIR \
    -ts 0.15 \
    -sw 0.01 \
    -temp  4.0 \
    -alpha 0.7 \
    --teacher_arch resnet50 \
    --teacher_pretrained \
    --label_smoothing 0.1 \
    -gpu 0 \
    -s

!cp -a save_ImageNet_model "$SAVE_DIR/save_ImageNet_model_kd"

## Cell 10 -- Evaluate All Saved Models

In [ ]:
import glob

variants = [
    ("Baseline PG",      0, f"{SAVE_DIR}/save_ImageNet_model_baseline"),
    ("Adaptive PG",      1, f"{SAVE_DIR}/save_ImageNet_model_adaptive"),
    ("Adaptive PG + KD", 2, f"{SAVE_DIR}/save_ImageNet_model_kd"),
]

for name, mid, path in variants:
    ckpts = sorted(glob.glob(os.path.join(path, "model_*.pt")))
    if not ckpts:
        print(f"\n[SKIP] {name} -- no checkpoint found in {path}")
        continue
    ckpt = ckpts[0]
    print(f"\n{'='*60}")
    print(f"Evaluating: {name}  (checkpoint: {ckpt})")
    print(f"{'='*60}")
    !python imagenet.py -id $mid -t -r "$ckpt" -d $IMAGENET_DIR -b 128 -gpu 0

## Cell 11 -- Resume Training After Session Timeout

Kaggle GPU sessions last up to 12 h.  
Re-run Cells 1-6 (fast -- repo is cached, val is already reorganised), then use this cell.  
Edit the three variables below to match the variant you were training.

In [ ]:
# --- edit these three lines to match your variant ---
MODEL_ID   = 1
CKPT_MODEL = f"{SAVE_DIR}/save_ImageNet_model_adaptive/model_adaptive-pg.pt"
CKPT_STATE = f"{SAVE_DIR}/save_ImageNet_model_adaptive/states_adaptive-pg.pt"
# ----------------------------------------------------

os.chdir(REPO_DIR)

!python imagenet.py \
    -id $MODEL_ID \
    -e  120 \
    -b  128 \
    -lr 5e-4 \
    -wd 1e-5 \
    -d  $IMAGENET_DIR \
    -r  "$CKPT_MODEL" \
    -l  "$CKPT_STATE" \
    -ts 0.15 \
    -sw 0.01 \
    --label_smoothing 0.1 \
    -gpu 0 \
    -s

---
## Reference

### Hyperparameters for good accuracy on the subset

| Parameter | Value | Notes |
|-----------|-------|-------|
| Epochs | 120 | Full schedule for the subset |
| Batch size | 128 (baseline/adaptive), 64 (KD) | P100/T4 safe |
| Learning rate | 5e-4, linear decay | |
| Weight decay | 1e-5 | |
| Label smoothing | 0.1 | |
| Target sparsity | 0.15 | 15% channels get 2-bit |
| Sparsity weight | 0.01 | |
| KD temperature | 4.0 | |
| KD alpha | 0.7 | |
| Teacher | ResNet-50 pretrained | torchvision weights |

### Accuracy note

This Kaggle dataset is a subset, so its accuracy numbers are useful for comparing:
- baseline vs adaptive PG
- no-KD vs KD
- whether the new pushed changes help

They are **not** directly comparable to full ImageNet paper numbers.

### Kaggle runtime notes

- GPU quota: 30 h/week (T4 x2) or 20 h/week (P100)
- Session limit: 12 h
- The subset is much smaller than full ImageNet, so each epoch should be noticeably faster
- 120 epochs may still need multiple sessions; use Cell 11 to resume
- `/kaggle/working/` persists across sessions when Persistence is enabled